# Projet RI - TREC 9

In [1]:
import pandas as pd                                                           
import numpy as np                 
import os                                                                     
import sys                                                                    
                                                                                                                                                                                                                    
import pyterrier as pt
from src.retrieval import *                                                   
from src.utils import *
from src.importance import *                                                  
from src.xquad import *
from src.subqueries import *     

In [2]:
init_pt()

Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/ziyuezhang/Desktop/projet2/rital_projet2_2026/src/retrieval.py:12: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [4]:
# Vaswani                                                                     
dataset_vaswani = pt.get_dataset("vaswani")
qrels_vaswani = dataset_vaswani.get_qrels()                                   
   
  # TREC 9                                                                      
dataset_wt10g = pt.get_dataset("trec-wt10g")
topics_trec9 = dataset_wt10g.get_topics("trec9")                              
qrels_trec9 = pd.read_csv(
    "qrels.trec9.main_web.gz",                                                
    sep=" ", header=None,                                                     
    names=["qid", "iteration", "docno", "relevance"]
)                                                                             
qrels_trec9["qid"] = qrels_trec9["qid"].astype(str)
topics_trec9["qid"] = topics_trec9["qid"].astype(str)                         
                  
print("Vaswani qrels:", qrels_vaswani.shape)                                  
print("TREC 9 topics:", topics_trec9.shape)
print("TREC 9 qrels:", qrels_trec9.shape)        

Vaswani qrels: (2083, 3)
TREC 9 topics: (50, 2)
TREC 9 qrels: (70070, 4)


In [5]:
import os                                                                     
                  
run_files = {                                                                 
      "baseline_bm25": "baseline_bm25.csv",
      "dummy_uniform": "xquad_dummy_uniform.csv",            
      "dummy_n": "xquad_dummy_n.csv",
      "dummy_redde": "xquad_dummy_redde.csv",                
      "dummy_crcs": "xquad_dummy_crcs.csv",                  
      "kmeans_uniform": "xquad_kmeans_uniform.csv",          
      "kmeans_n": "xquad_kmeans_n.csv",                      
      "kmeans_redde": "xquad_kmeans_redde.csv",
      "kmeans_crcs": "xquad_kmeans_crcs.csv",                
}               
                                                                                
runs = {}       
for name, path in run_files.items():
    runs[name] = pd.read_csv(path)                                            
    print(name, runs[name].shape, sorted(runs[name]['qid'].unique()))


baseline_bm25 (500, 5) [1, 2, 3, 4, 5]
dummy_uniform (500, 5) [1, 2, 3, 4, 5]
dummy_n (500, 5) [1, 2, 3, 4, 5]
dummy_redde (500, 5) [1, 2, 3, 4, 5]
dummy_crcs (500, 5) [1, 2, 3, 4, 5]
kmeans_uniform (500, 5) [1, 2, 3, 4, 5]
kmeans_n (500, 5) [1, 2, 3, 4, 5]
kmeans_redde (500, 5) [1, 2, 3, 4, 5]
kmeans_crcs (500, 5) [1, 2, 3, 4, 5]


In [7]:
metrics = ['map', 'ndcg_cut_10', 'P_10']                                      
                                                                                
results = {}                                                                  
for name, run in runs.items():                                                
    res = pt.Evaluate(run, qrels_vaswani, metrics=metrics)                    
    results[name] = res                                                       
   
results_df = pd.DataFrame(results).T                                          
print(results_df)

                     map  ndcg_cut_10      P_10
baseline_bm25   0.010155     0.018622  0.012903
dummy_uniform   0.010155     0.018622  0.012903
dummy_n         0.010155     0.018622  0.012903
dummy_redde     0.010155     0.018622  0.012903
dummy_crcs      0.010155     0.018622  0.012903
kmeans_uniform  0.007207     0.013092  0.010753
kmeans_n        0.007207     0.013092  0.010753
kmeans_redde    0.007016     0.012999  0.010753
kmeans_crcs     0.007262     0.014146  0.011828


In [8]:
qrels_trec9 = pd.read_csv(                                                    
    "qrels.trec9.main_web.gz",                             
    sep=" ", header=None,                                                     
    names=["qid", "iteration", "docno", "relevance"]                          
)                                                                             
                                                                                
  # Topics déjà chargés, vérifier                                               
print(topics_trec9.head(3))
print(qrels_trec9.head(3))                                                    
print("Nb queries:", qrels_trec9["qid"].nunique())  

   qid                          query
0  451         What is a Bengals cat?
1  452  do beavers live in salt water
2  453                         hunger
   qid  iteration          docno  relevance
0  451          0  WTX001-B06-78          0
1  451          0  WTX001-B08-58          0
2  451          0  WTX001-B17-53          0
Nb queries: 50


In [11]:
# Pour chaque query, trier les documents par pertinence décroissante          
pseudo_results = []
                                                                                
for qid in qrels_trec9["qid"].unique():
    df_q = qrels_trec9[qrels_trec9["qid"] == qid].copy()                      
    df_q = df_q.sample(frac=1, random_state=42).reset_index(drop=True)        
    df_q["rank"] = df_q.index + 1
    df_q["score"] = (len(df_q) - df_q.index).astype(float)                    
    pseudo_results.append(df_q[["qid", "docno", "rank", "score"]])            
                                                                                
pseudo_df = pd.concat(pseudo_results, ignore_index=True)                      
print(pseudo_df.shape)
print(pseudo_df.head(5)) 

(70070, 4)
   qid           docno  rank   score
0  451  WTX055-B49-318     1  1174.0
1  451  WTX056-B17-227     2  1173.0
2  451  WTX071-B31-167     3  1172.0
3  451   WTX033-B49-39     4  1171.0
4  451   WTX080-B43-58     5  1170.0


In [12]:
pseudo_df["qid"] = pseudo_df["qid"].astype(str)

In [13]:
from src.utils import results_to_doclist, results_to_score_dict,ranking_to_run_df                                                             
from src.subqueries import generate_dummy_subqueries                          
from src.importance import importance_uniform,importance_n, importance_redde,importance_crcs                  
from src.xquad import xquad_rerank                                            
  
                                                       
qid = "451"       
query = topics_trec9[topics_trec9["qid"] == qid]["query"].values[0]           
df_q = pseudo_df[pseudo_df["qid"] == qid].copy()
                                                                                
docs_ranked = results_to_doclist(df_q)
r_query = results_to_score_dict(df_q)                                         
                                                                                
# Dummy subqueries
subqueries = generate_dummy_subqueries(query)                                 
                  
# mêmes scores que r_query pour chaque sous-requête           
r_sub = {doc: {sq: r_query.get(doc, 0) for sq in subqueries} for doc in docs_ranked}                                                                  
                                                                  
imp = importance_uniform(subqueries)

result = xquad_rerank(docs_ranked, subqueries, r_query, r_sub, imp, tau=20,omega=0.5)                                                                    
  
print("Query:", query)                                                        
print("Baseline top5 :", docs_ranked[:5])
print("xQuAD top5    :", result[:5])

Query: What is a Bengals cat?
Baseline top5 : ['WTX055-B49-318', 'WTX056-B17-227', 'WTX071-B31-167', 'WTX033-B49-39', 'WTX080-B43-58']
xQuAD top5    : ['WTX055-B49-318', 'WTX056-B17-227', 'WTX071-B31-167', 'WTX033-B49-39', 'WTX080-B43-58']


In [15]:
from src.utils import results_to_doclist, results_to_score_dict,ranking_to_run_df
from src.subqueries import generate_dummy_subqueries
from src.importance import importance_uniform, importance_n, importance_redde,importance_crcs
from src.xquad import xquad_rerank

all_runs_trec9 = {"baseline": [], "xquad_uniform": [], "xquad_n": [],"xquad_redde": [], "xquad_crcs": []}

for _, row in topics_trec9.iterrows():
    qid = row["qid"]
    query = row["query"]
    df_q = pseudo_df[pseudo_df["qid"] == qid].copy()
    if df_q.empty:
        continue
    docs_ranked = results_to_doclist(df_q)
    r_query = results_to_score_dict(df_q)
    subqueries = generate_dummy_subqueries(query)
    r_sub = {doc: {sq: r_query.get(doc, 0) for sq in subqueries} for doc in docs_ranked}
    n_qi = {sq: len(docs_ranked) for sq in subqueries}

    all_runs_trec9["baseline"].append(ranking_to_run_df(qid, query,docs_ranked[:100]))
    all_runs_trec9["xquad_uniform"].append(ranking_to_run_df(qid, query,xquad_rerank(docs_ranked, subqueries, r_query, r_sub,importance_uniform(subqueries), tau=100, omega=0.5)))
    all_runs_trec9["xquad_n"].append(ranking_to_run_df(qid, query,xquad_rerank(docs_ranked, subqueries, r_query, r_sub,importance_n(n_qi), tau=100, omega=0.5)))
    all_runs_trec9["xquad_redde"].append(ranking_to_run_df(qid, query,xquad_rerank(docs_ranked, subqueries, r_query, r_sub,importance_redde(docs_ranked, subqueries, r_query, r_sub, n_qi), tau=100,omega=0.5)))
    all_runs_trec9["xquad_crcs"].append(ranking_to_run_df(qid, query,xquad_rerank(docs_ranked, subqueries, r_query, r_sub,importance_crcs(docs_ranked, subqueries, r_sub, n_qi), tau=100, omega=0.5)))

for name in all_runs_trec9:
    all_runs_trec9[name] = pd.concat(all_runs_trec9[name], ignore_index=True)
    all_runs_trec9[name]["qid"] = all_runs_trec9[name]["qid"].astype(str)
    print(name, all_runs_trec9[name].shape)

baseline (5000, 5)
xquad_uniform (5000, 5)
xquad_n (5000, 5)
xquad_redde (5000, 5)
xquad_crcs (5000, 5)


In [16]:
qrels_for_eval = qrels_trec9[["qid", "docno", "relevance"]].copy()            
qrels_for_eval.columns = ["qid", "docno", "label"]                            
qrels_for_eval["qid"] = qrels_for_eval["qid"].astype(str)                     
                                                                                
metrics = ["map", "ndcg_cut_10", "P_10"]                                      
                  
results_trec9 = {}                                                            
for name, run in all_runs_trec9.items():
    results_trec9[name] = pt.Evaluate(run, qrels_for_eval, metrics=metrics)   
                                                                                
results_trec9_df = pd.DataFrame(results_trec9).T
print(results_trec9_df) 

                    map  ndcg_cut_10   P_10
baseline       0.006989     0.022998  0.036
xquad_uniform  0.006989     0.022998  0.036
xquad_n        0.006989     0.022998  0.036
xquad_redde    0.006989     0.022998  0.036
xquad_crcs     0.006989     0.022998  0.036


In [ ]:
#trec-2002 n'est pas disponible

In [17]:
dataset_gov = pt.get_dataset("irds:gov/trec-web-2002")
topics_gov = dataset_gov.get_topics("title")                                  
qrels_gov = dataset_gov.get_qrels()
                                                                                
print("Topics:", topics_gov.shape)
print(topics_gov.head(3))                                                     
print("Qrels:", qrels_gov.shape)
print(qrels_gov.head(3))     

[INFO] [starting] https://trec.nist.gov/data/qrels_eng/qrels.distillation.txt.gz
[INFO] [finished] https://trec.nist.gov/data/qrels_eng/qrels.distillation.txt.gz: [00:00] [403kB] [1.26MB/s]


Topics: (50, 2)
   qid                           query
0  551           intellectual property
1  552       Foods for cancer patients
2  553  federal funding mental illness
Qrels: (56650, 4)
   qid           docno  label iteration
0  551  G14-77-3709129      0         0
1  551  G08-22-1623396      0         0
2  551  G00-62-3810067      0         0


In [18]:
try:                                                                          
    corpus_iter = dataset_gov.get_corpus_iter()
    sample = next(iter(corpus_iter))
    print(sample)
except Exception as e:                                                        
    print("Erreur:", e)

[INFO] [starting] building docstore
[INFO] GOV is available by hard drive from UoG here: <http://ir.dcs.gla.ac.uk/test_collections/access_to_data.html>
More details about the procedure can be found here: <https://ir-datasets.com/gov.html#DataAccess>.
Link the GOV source files here: /Users/ziyuezhang/.ir_datasets/gov/corpus
Should contain G00, G01, G02, ...
docs_iter:   0%|                                   | 0/1247753 [00:00<?, ?doc/s]

Erreur: /Users/ziyuezhang/.ir_datasets/gov/corpus



[INFO] [error] docs_iter: [00:00] [0doc] [?doc/s]
[INFO] [error] building docstore s]
